# Bài 6 (trang PDF 45, Groceries) – Câu 4

**Sinh viên:** Trần Chí Nguyên  
**MSSV:** 6451071052

Notebook chạy độc lập trên Google Colab. Dữ liệu công khai được tải tự động khi chạy cell đầu tiên.

In [ ]:
import csv, requests, time, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("Trần Chí Nguyên - 6451071052")
FILE_ID = "1UIeIAeGPrmXaSsFOtfhw9h5gK4RPkseI"
def load_groceries():
    path = Path("/content/groceries.csv")
    if not path.exists():
        response = requests.get("https://drive.google.com/uc", params={"export": "download", "id": FILE_ID}, timeout=90)
        response.raise_for_status()
        path.write_bytes(response.content)
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return [[item.strip() for item in row if item.strip()] for row in csv.reader(handle)]
transactions = load_groceries()
print(f"Số giao dịch: {len(transactions):,}; số mặt hàng duy nhất: {len(set(x for t in transactions for x in t)):,}")

## Thực hiện yêu cầu

Cell dưới đây chứa lời giải cho câu này.

In [ ]:
%pip -q install mlxtend
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
encoder = TransactionEncoder(); encoded = encoder.fit(transactions).transform(transactions, sparse=True)
onehot = pd.DataFrame.sparse.from_spmatrix(encoded, columns=encoder.columns_)
MIN_SUPPORTS = np.round(np.arange(0.0001, 0.00101, 0.0001), 4).tolist()
MAX_ITEMSET_LENGTH = 3
apriori_results = []
for min_sup in MIN_SUPPORTS:
    start = time.perf_counter()
    patterns = apriori(onehot, min_support=min_sup, use_colnames=True, low_memory=True, max_len=MAX_ITEMSET_LENGTH)
    apriori_results.append({"min_support": min_sup, "patterns": len(patterns), "seconds": time.perf_counter() - start})
apriori_results = pd.DataFrame(apriori_results)
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(apriori_results["min_support"] * 100, apriori_results["seconds"], marker="o", color="royalblue")
ax1.set(xlabel="min_sup (%)", ylabel="Thời gian (giây)", title="Apriori trên Groceries")
ax2 = ax1.twinx(); ax2.plot(apriori_results["min_support"] * 100, apriori_results["patterns"], marker="s", color="darkorange")
ax2.set_ylabel("Số tập phổ biến (tối đa 3 mặt hàng)"); plt.tight_layout(); plt.show()
display(apriori_results)